## Initialize


### Import necessary packages


In [1]:
import numpy as np
import marimo as mo
import polars as pl
from pathlib import Path
import re

### Setup global variables


In [2]:
DATA_DIR = Path("../Dataset Round 01/Data")
OUTPUT_DIR = Path("../Dataset Round 01/Cleaned")

## Preprocessing


### Brandhealth Dataset


#### Read the dataset


In [3]:
raw_brand_health_df = pl.read_csv(
    DATA_DIR / "Brandhealth.csv",
    separator=";",
    schema_overrides={
        "PPA": float,
        "Fre#visit": int,
        "NPS#P3M": int,
        "Spending": int,
        "Spending_use": int,
    },
).unique()
raw_brand_health_df

ID,Year,City,Brand,Spontaneous,Awareness,Trial,P3M,P1M,Comprehension,Brand_Likability,Weekly,Daily,Fre#visit,PPA,Spending,Segmentation,NPS#P3M,NPS#P3M#Group,Spending_use
i64,i64,str,str,str,str,str,str,str,str,str,str,str,i64,f64,i64,str,i64,str,i64
435614,2018,"""Hà Nội""","""Other 1""","""Other 1""","""Other 1""","""Other 1""","""Other 1""","""Other 1""",null,null,null,null,2,30.0,60,"""Seg.02 - Mass Asp (VND 25K - V…",10,"""Promoter""",60
836136,2019,"""Hồ Chí Minh""","""Trung Nguyên""",null,"""Trung Nguyên""",null,null,null,"""Know a little""",null,null,null,null,null,null,null,null,null,null
369436,2018,"""Hồ Chí Minh""","""Street / Half street coffee (i…","""Street / Half street coffee (i…","""Street / Half street coffee (i…","""Street / Half street coffee (i…","""Street / Half street coffee (i…","""Street / Half street coffee (i…",null,null,"""Street / Half street coffee (i…",null,10,10.0,100,"""Seg.01 - Mass (<VND 25K)""",5,"""Detractor""",100
369434,2018,"""Hồ Chí Minh""","""Highlands Coffee""","""Highlands Coffee""","""Highlands Coffee""","""Highlands Coffee""",null,null,null,null,null,null,null,null,null,null,null,null,null
460180,2018,"""Hà Nội""","""Urban Station""",null,"""Urban Station""",null,null,null,null,null,null,null,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
739501,2019,"""Hồ Chí Minh""","""Viva Star""",null,"""Viva Star""",null,null,null,"""Know a little""",null,null,null,null,null,null,null,null,null,null
89965,2017,"""Hồ Chí Minh""","""Trung Nguyên""",null,"""Trung Nguyên""","""Trung Nguyên""",null,null,null,null,null,null,null,null,null,null,null,null,null
453737,2018,"""Nha Trang""","""Other 1""","""Other 1""","""Other 1""","""Other 1""","""Other 1""","""Other 1""",null,null,"""Other 1""",null,4,20.0,80,"""Seg.01 - Mass (<VND 25K)""",8,"""Passive""",80


#### Check if each ID applies for one year and one city


In [4]:
raw_brand_health_df.unique("ID").height, raw_brand_health_df.unique(
    ["ID", "Year", "City"]
).height

(11761, 11761)

The number of unique rows with `ID`, and the tuple (`ID`, `Year`, `City`) is the same, suggesting that `Year` and `City` can be derived from a unique `ID`.


#### Remove redundant columns


Check if `Spending` is the same as `Spending_use`


In [5]:
raw_brand_health_df.filter(pl.col("Spending") != pl.col("Spending_use"))

ID,Year,City,Brand,Spontaneous,Awareness,Trial,P3M,P1M,Comprehension,Brand_Likability,Weekly,Daily,Fre#visit,PPA,Spending,Segmentation,NPS#P3M,NPS#P3M#Group,Spending_use
i64,i64,str,str,str,str,str,str,str,str,str,str,str,i64,f64,i64,str,i64,str,i64


There are no results. Indicating that they are the same. Therefore we will remove $Spending_use$.

We will also remove `City` and `Year` for obvious reasons.


In [6]:
brand_health_df = raw_brand_health_df.drop(["City", "Year", "Spending_use"])

#### Impute missing values


In [7]:
brand_health_df.null_count()

ID,Brand,Spontaneous,Awareness,Trial,P3M,P1M,Comprehension,Brand_Likability,Weekly,Daily,Fre#visit,PPA,Spending,Segmentation,NPS#P3M,NPS#P3M#Group
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,43426,114,27089,45570,55020,48073,64088,61037,66798,55087,60346,60346,60346,52814,52814


There are a lot of missing values. Let's try to impute them.


In [8]:
mo.md(
    rf"""For `Comprehension`, since there is no way to deduce its value, and it takes up a large proportion of the dataset ({100 * brand_health_df["Comprehension"].null_count() / brand_health_df.height:.2f}%), we can consider droping the column."""
)

_md()

In [ ]:
brand_health_df = brand_health_df.drop("Comprehension")

For statistical attributes (`Fre`, `PPA`, `NPS`), missing values could indicate that the value is 0.


In [ ]:
NUMERICAL_ATTRIBUTES = ["Fre#visit", "PPA", "NPS#P3M", "Spending"]

brand_health_df = brand_health_df.with_columns(
    pl.col(NUMERICAL_ATTRIBUTES).fill_null(0)
)
brand_health_df.select(NUMERICAL_ATTRIBUTES).null_count()

Fre#visit,PPA,NPS#P3M,Spending
u32,u32,u32,u32
0,0,0,0


For categorical attributes (`Segmentation` and `NPS`), the value can be imputed using the corresponding attributes.

Let's first check for all of the possible attributes:


In [ ]:
brand_health_df["Segmentation"].unique().sort(), brand_health_df[
    "NPS#P3M#Group"
].unique().sort()

(shape: (5,)
 Series: 'Segmentation' [str]
 [
 	null
 	"Seg.01 - Mass (<VND 25K)"
 	"Seg.02 - Mass Asp (VND 25K - V…
 	"Seg.03 - Premium (VND 60K - VN…
 	"Seg.04 - Super Premium (VND 10…
 ],
 shape: (4,)
 Series: 'NPS#P3M#Group' [str]
 [
 	null
 	"Detractor"
 	"Passive"
 	"Promoter"
 ])

We then create a mapper function for each category:


In [12]:
def segmenatation_map(ppa: int) -> str:
    if ppa < 25:
        return "Seg.01 - Mass (<VND 25K)"
    if ppa < 60:
        return "Seg.02 - Mass Asp (VND 25K - VND 59K)"
    if ppa < 100:
        return "Seg.03 - Premium (VND 60K - VND 99K)"
    return "Seg.04 - Super Premium (VND 100K+)"


def nps_map(nps: int) -> str:
    if nps < 7:
        return "Detractor"
    if nps < 9:
        return "Passive"
    return "Promoter"

We use the functions to impute the categorical attributes


In [ ]:
CATEGORICAL_MAPPING = [
    ("Segmentation", "PPA", segmenatation_map),
    ("NPS#P3M#Group", "NPS#P3M", nps_map),
]

brand_health_df = brand_health_df.with_columns(
    [
        pl.when(pl.col(category).is_null())
        .then(pl.col(value).map_elements(func, return_dtype=str))
        .otherwise(pl.col(category))
        .alias(category)
        for category, value, func in CATEGORICAL_MAPPING
    ]
)
brand_health_df.select(NUMERICAL_ATTRIBUTES).null_count()

Fre#visit,PPA,NPS#P3M,Spending
u32,u32,u32,u32
0,0,0,0


For `Spontaneous`, `Awareness`, `Brand_Likability`, and the boolean columns; their values, if not missing, is one of `Brand`:


In [ ]:
BOOLEAN_COLUMNS = [
    "Spontaneous",
    "Awareness",
    "Trial",
    "P3M",
    "P1M",
    "Brand_Likability",
]

for column in ["Brand"] + BOOLEAN_COLUMNS:
    print(brand_health_df[column].drop_nulls().unique().sort())

shape: (36,)
Series: 'Brand' [str]
[
	"Aha Cafe"
	"BonPas"
	"Cheese Coffee"
	"Coffee Bean & Tea Leaf"
	"Cộng Cà Phê"
	…
	"Thức Coffee"
	"Trung Nguyên"
	"Urban Station"
	"Viva Star"
	"Đen Đá"
]
shape: (37,)
Series: 'Spontaneous' [str]
[
	"Aha Cafe"
	"BonPas"
	"Cheese Coffee"
	"Coffee Bean & Tea Leaf"
	"Cộng Cà Phê"
	…
	"Thức Coffee"
	"Trung Nguyên"
	"Urban Station"
	"Viva Star"
	"Đen Đá"
]
shape: (37,)
Series: 'Awareness' [str]
[
	"Aha Cafe"
	"BonPas"
	"Cheese Coffee"
	"Coffee Bean & Tea Leaf"
	"Cộng Cà Phê"
	…
	"Thức Coffee"
	"Trung Nguyên"
	"Urban Station"
	"Viva Star"
	"Đen Đá"
]
shape: (37,)
Series: 'Trial' [str]
[
	"Aha Cafe"
	"BonPas"
	"Cheese Coffee"
	"Coffee Bean & Tea Leaf"
	"Cộng Cà Phê"
	…
	"Thức Coffee"
	"Trung Nguyên"
	"Urban Station"
	"Viva Star"
	"Đen Đá"
]
shape: (37,)
Series: 'P3M' [str]
[
	"Aha Cafe"
	"BonPas"
	"Cheese Coffee"
	"Coffee Bean & Tea Leaf"
	"Cộng Cà Phê"
	…
	"Thức Coffee"
	"Trung Nguyên"
	"Urban Station"
	"Viva Star"
	"Đen Đá"
]
shape: (36,)
Series: 'P1M' 

Therefore, we use that rule to impute the columns.


In [ ]:
brand_health_df = brand_health_df.with_columns(
    [
        pl.when(pl.col(attribute) == pl.col("Brand"))
        .then(pl.lit(True))
        .otherwise(pl.lit(False))
        .alias(attribute)
        for attribute in BOOLEAN_COLUMNS
    ]
)
brand_health_df.select(BOOLEAN_COLUMNS).null_count()

Spontaneous,Awareness,Trial,P3M,P1M,Brand_Likability
u32,u32,u32,u32,u32,u32
0,0,0,0,0,0


For `Weekly` and `Daily`, we also have to check for `Applicable`.


In [ ]:
DAYS_OF_WEEK_COL = ["Weekly", "Daily"]

brand_health_df[DAYS_OF_WEEK_COL].filter(
    (pl.col("Weekly") == "Applicable") | (pl.col("Daily") == "Applicable")
)

Weekly,Daily
str,str


Since there is no brand named `Not Applicable`, we can use the same method as above.


In [ ]:
brand_health_df = brand_health_df.with_columns(
    [
        pl.when(pl.col(attribute) == pl.col("Brand"))
        .then(pl.lit("Applicable"))
        .otherwise(pl.lit("Not Applicable"))
        .alias(attribute)
        for attribute in ["Weekly", "Daily"]
    ]
)
brand_health_df.select(["Weekly", "Daily"]).null_count()

Weekly,Daily
u32,u32
0,0


It seems that we are done. Let's check the result.


In [ ]:
brand_health_df.null_count()

ID,Brand,Spontaneous,Awareness,Trial,P3M,P1M,Brand_Likability,Weekly,Daily,Fre#visit,PPA,Spending,Segmentation,NPS#P3M,NPS#P3M#Group
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


#### Logical Correction

We have to make sure that:

- If $\text{Spontaneous} = \text{True}$, then $\text{Awareness} = \text{True}$.

- If $\text{Trial} = \text{True}$, then $\text{Awareness} = True$.

- If $\text{P1M} = \text{True}$, then $\text{P3M} = \text{True}$.

- If $\text{Daily} = \text{True}$, then $\text{Weekly} = \text{True}$.

- $\text{Spending} = 0 \Leftrightarrow \text{Fre} = 0$.

- The formula for $\text{PPA}$ is correct.

- `Segmentation` and `NPS` is grouped correctly.


First task:


In [ ]:
brand_health_df = brand_health_df.filter(
    (pl.col("Awareness") == True) | (pl.col("Spontaneous") == False)
)
brand_health_df.filter((pl.col("Awareness") == False) & (pl.col("Spontaneous") == True))

ID,Brand,Spontaneous,Awareness,Trial,P3M,P1M,Brand_Likability,Weekly,Daily,Fre#visit,PPA,Spending,Segmentation,NPS#P3M,NPS#P3M#Group
i64,str,bool,bool,bool,bool,bool,bool,str,str,i64,f64,i64,str,i64,str
344634,"""Other 3""",true,false,true,false,false,false,"""Not Applicable""","""Not Applicable""",0,0.0,0,"""Seg.01 - Mass (<VND 25K)""",0,"""Detractor"""
368681,"""Other 1""",true,false,false,false,false,false,"""Not Applicable""","""Not Applicable""",0,0.0,0,"""Seg.01 - Mass (<VND 25K)""",0,"""Detractor"""
348222,"""Other 2""",true,false,true,true,true,false,"""Applicable""","""Applicable""",20,8.0,160,"""Seg.01 - Mass (<VND 25K)""",9,"""Promoter"""
353401,"""Other 2""",true,false,true,true,false,false,"""Not Applicable""","""Not Applicable""",0,0.0,0,"""Seg.01 - Mass (<VND 25K)""",7,"""Passive"""
378613,"""Other 1""",true,false,true,true,true,false,"""Applicable""","""Not Applicable""",8,25.0,200,"""Seg.02 - Mass Asp (VND 25K - V…",10,"""Promoter"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
344634,"""Other 1""",true,false,true,true,true,false,"""Applicable""","""Applicable""",30,10.0,300,"""Seg.01 - Mass (<VND 25K)""",9,"""Promoter"""
92157,"""Indepedent Cafe""",true,false,true,true,true,false,"""Applicable""","""Not Applicable""",5,0.0,0,"""Seg.01 - Mass (<VND 25K)""",0,"""Detractor"""
93387,"""Indepedent Cafe""",true,false,false,false,false,false,"""Not Applicable""","""Not Applicable""",0,0.0,0,"""Seg.01 - Mass (<VND 25K)""",0,"""Detractor"""


Second task:


In [ ]:
brand_health_df = brand_health_df.filter(
    (pl.col("Awareness") == True) | (pl.col("Trial") == False)
)
brand_health_df.filter((pl.col("Awareness") == False) & (pl.col("Trial") == True))

ID,Brand,Spontaneous,Awareness,Trial,P3M,P1M,Brand_Likability,Weekly,Daily,Fre#visit,PPA,Spending,Segmentation,NPS#P3M,NPS#P3M#Group
i64,str,bool,bool,bool,bool,bool,bool,str,str,i64,f64,i64,str,i64,str
380067,"""Street / Half street coffee (i…",false,false,true,false,false,false,"""Not Applicable""","""Not Applicable""",0,0.0,0,"""Seg.01 - Mass (<VND 25K)""",0,"""Detractor"""
344635,"""Other 3""",false,false,true,false,false,false,"""Not Applicable""","""Not Applicable""",0,0.0,0,"""Seg.01 - Mass (<VND 25K)""",0,"""Detractor"""
368681,"""Other 3""",false,false,true,true,true,false,"""Not Applicable""","""Not Applicable""",1,25.0,25,"""Seg.02 - Mass Asp (VND 25K - V…",7,"""Passive"""
371738,"""Street / Half street coffee (i…",false,false,true,false,false,false,"""Not Applicable""","""Not Applicable""",0,0.0,0,"""Seg.01 - Mass (<VND 25K)""",0,"""Detractor"""
117710,"""Indepedent Cafe""",false,false,true,true,true,false,"""Not Applicable""","""Not Applicable""",3,0.0,0,"""Seg.01 - Mass (<VND 25K)""",0,"""Detractor"""
383551,"""Street / Half street coffee (i…",false,false,true,false,false,false,"""Not Applicable""","""Not Applicable""",0,0.0,0,"""Seg.01 - Mass (<VND 25K)""",0,"""Detractor"""
348224,"""Other 2""",false,false,true,true,true,false,"""Not Applicable""","""Not Applicable""",1,20.0,20,"""Seg.01 - Mass (<VND 25K)""",8,"""Passive"""
131558,"""Indepedent Cafe""",false,false,true,true,true,false,"""Applicable""","""Applicable""",28,0.0,0,"""Seg.01 - Mass (<VND 25K)""",0,"""Detractor"""
367856,"""Other 1""",false,false,true,false,false,false,"""Not Applicable""","""Not Applicable""",0,0.0,0,"""Seg.01 - Mass (<VND 25K)""",0,"""Detractor"""


Third task:


In [ ]:
brand_health_df = brand_health_df.filter(
    (pl.col("P3M") == True) | (pl.col("P1M") == False)
)
brand_health_df.filter((pl.col("P3M") == False) & (pl.col("P1M") == True))

ID,Brand,Spontaneous,Awareness,Trial,P3M,P1M,Brand_Likability,Weekly,Daily,Fre#visit,PPA,Spending,Segmentation,NPS#P3M,NPS#P3M#Group
i64,str,bool,bool,bool,bool,bool,bool,str,str,i64,f64,i64,str,i64,str
109201,"""Cộng Cà Phê""",true,true,false,false,true,false,"""Applicable""","""Not Applicable""",4,0.0,0,"""Seg.01 - Mass (<VND 25K)""",0,"""Detractor"""
112543,"""Street / Half street coffee (i…",false,true,true,false,true,false,"""Not Applicable""","""Not Applicable""",1,0.0,0,"""Seg.01 - Mass (<VND 25K)""",0,"""Detractor"""
97503,"""Cộng Cà Phê""",true,true,false,false,true,false,"""Applicable""","""Not Applicable""",4,0.0,0,"""Seg.01 - Mass (<VND 25K)""",0,"""Detractor"""


Third task:


In [ ]:
brand_health_df.filter((pl.col("P3M") == False) & (pl.col("P1M") == True))

ID,Brand,Spontaneous,Awareness,Trial,P3M,P1M,Brand_Likability,Weekly,Daily,Fre#visit,PPA,Spending,Segmentation,NPS#P3M,NPS#P3M#Group
i64,str,bool,bool,bool,bool,bool,bool,str,str,i64,f64,i64,str,i64,str


Fourth task:


In [ ]:
brand_health_df_9 = brand_health_df.filter(
    (pl.col("Fre#visit") == 0) ^ (pl.col("Spending") != 0)
)
brand_health_df.filter((pl.col("Fre#visit") != 0) ^ (pl.col("Spending") != 0))

ID,Brand,Spontaneous,Awareness,Trial,P3M,P1M,Brand_Likability,Weekly,Daily,Fre#visit,PPA,Spending,Segmentation,NPS#P3M,NPS#P3M#Group
i64,str,bool,bool,bool,bool,bool,bool,str,str,i64,f64,i64,str,i64,str
122191,"""Indepedent Cafe""",true,true,true,true,true,false,"""Applicable""","""Not Applicable""",6,0.0,0,"""Seg.01 - Mass (<VND 25K)""",0,"""Detractor"""
138379,"""Highlands Coffee""",true,true,true,true,true,false,"""Not Applicable""","""Not Applicable""",2,0.0,0,"""Seg.01 - Mass (<VND 25K)""",9,"""Promoter"""
134968,"""Indepedent Cafe""",true,true,true,true,true,false,"""Applicable""","""Not Applicable""",14,0.0,0,"""Seg.01 - Mass (<VND 25K)""",0,"""Detractor"""
108892,"""Street / Half street coffee (i…",true,true,true,true,true,false,"""Applicable""","""Not Applicable""",4,0.0,0,"""Seg.01 - Mass (<VND 25K)""",0,"""Detractor"""
123637,"""Indepedent Cafe""",true,true,true,true,true,false,"""Applicable""","""Not Applicable""",4,0.0,0,"""Seg.01 - Mass (<VND 25K)""",0,"""Detractor"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
125945,"""Indepedent Cafe""",true,true,true,true,true,false,"""Applicable""","""Not Applicable""",4,0.0,0,"""Seg.01 - Mass (<VND 25K)""",0,"""Detractor"""
134864,"""Street / Half street coffee (i…",true,true,true,true,true,false,"""Applicable""","""Not Applicable""",4,0.0,0,"""Seg.01 - Mass (<VND 25K)""",0,"""Detractor"""
139005,"""Street / Half street coffee (i…",true,true,true,true,true,false,"""Not Applicable""","""Not Applicable""",2,0.0,0,"""Seg.01 - Mass (<VND 25K)""",0,"""Detractor"""


Fifth task:


In [ ]:
brand_health_df.filter(
    pl.col("PPA")
    != pl.when(pl.col("Fre#visit") == 0)
    .then(pl.lit(0))
    .otherwise((pl.col("Spending") / pl.col("Fre#visit")))
    .round(1)
)

ID,Brand,Spontaneous,Awareness,Trial,P3M,P1M,Brand_Likability,Weekly,Daily,Fre#visit,PPA,Spending,Segmentation,NPS#P3M,NPS#P3M#Group
i64,str,bool,bool,bool,bool,bool,bool,str,str,i64,f64,i64,str,i64,str


Sixth task:


In [ ]:
brand_health_df.filter(
    (
        pl.col("Segmentation")
        != pl.col("PPA").map_elements(segmenatation_map, return_dtype=str)
    )
    | (
        pl.col("NPS#P3M#Group")
        != pl.col("NPS#P3M").map_elements(nps_map, return_dtype=str)
    )
)

ID,Brand,Spontaneous,Awareness,Trial,P3M,P1M,Brand_Likability,Weekly,Daily,Fre#visit,PPA,Spending,Segmentation,NPS#P3M,NPS#P3M#Group
i64,str,bool,bool,bool,bool,bool,bool,str,str,i64,f64,i64,str,i64,str


#### Save the result


In [ ]:
brand_health_df.write_csv(OUTPUT_DIR / "Brand Health.csv")